In [1]:
import numpy as np
import scipy as sp

In [4]:
pushkin_dataset = "/home/fedor/scicodes/ml/nano-gpt-tutorial/data/Pushkin.txt"

with open(pushkin_dataset, 'r') as t:
    text = t.read()

In [5]:
print("length of the dataset in characters = ", len(text))

length of the dataset in characters =  609236


In [7]:
print(text[:1000])

Перевод

Скажите мне, почему «Похититель»
Освистан партером?
Увы, потому что бедный автор
Похитил его у Мольера.


Перевод

Пою сей бой, в котором Толи одолел,
Где погиб не один воин, где Поль отличился,
Николая Матюрена и прекрасную Нитуш,
Рука которой была трофеем ужасной стычки.



Стихотворения 1813 г



К Наталье

Так и мне узнать случилось,
Что за птица Купидон;
Сердце страстное пленилось;
Признаюсь – и я влюблен!
Пролетело счастья время,
Как, любви не зная бремя,
Я живал да попевал,
Как в театре и на балах,
На гуляньях иль в воксалах
Легким зефиром летал;
Как, смеясь во зло Амуру,
Я писал карикатуру
На любезный женской пол;
Но напрасно я смеялся,
Наконец и сам попался,
Сам, увы! с ума сошел.
Смехи, вольность – всё под лавку
Из Катонов я в отставку,
И теперь я – Селадон!
Миловидной жрицы Тальи
Видел прелести Натальи,
И уж в сердце – Купидон!

Так, Наталья! признаюся,
Я тобою полонен,
В первый раз еще, стыжуся,
В женски прелести влюблен.
Целый день, как ни верчуся
Лишь тобою занят

In [9]:
# For tockenixation -- extract unique charaters
chars = sorted(list(set(text)))
vocab_size = len(chars)
print(''.join(chars))
print("vocabulary size = ", vocab_size)

	
 !"'()*,-.0123456789:;?ABCDEFGHIJKLMNOPRSTVWXZ_abcdefghijklmnopqrstuvwxyz «»АБВГДЕЖЗИЙКЛМНОПРСТУФХЦЧШЩЭЮЯабвгдежзийклмнопрстуфхцчшщъыьэюяё–—„…
vocabulary size =  144


In [13]:
# Tokinizer here

stoi = {ch: i for i,ch in enumerate(chars)}
itos = {i: ch for i,ch in enumerate(chars)}

encode = lambda s: [stoi[c] for c in s]
decode = lambda l: ''.join([itos[i] for i in l])

print(encode("Тестовый текст!"))
print(decode(encode("Тестовый текст!")))

[96, 112, 124, 125, 121, 109, 134, 116, 2, 125, 112, 117, 124, 125, 3]
Тестовый текст!


In [15]:
#encode the entire dataset

import torch

data = torch.tensor(encode(text), dtype=torch.long)
print(data.shape, data.dtype)
print(data[:1000])

torch.Size([609236]) torch.int64
tensor([ 93, 112, 123, 112, 109, 121, 111,   1,   1,  95, 117, 107, 113, 115,
        125, 112,   2, 119, 120, 112,   9,   2, 122, 121, 130, 112, 119, 126,
          2,  76,  93, 121, 128, 115, 125, 115, 125, 112, 118, 135,  77,   1,
         92, 124, 109, 115, 124, 125, 107, 120,   2, 122, 107, 123, 125, 112,
        123, 121, 119,  24,   1,  97, 109, 134,   9,   2, 122, 121, 125, 121,
        119, 126,   2, 130, 125, 121,   2, 108, 112, 111, 120, 134, 116,   2,
        107, 109, 125, 121, 123,   1,  93, 121, 128, 115, 125, 115, 118,   2,
        112, 110, 121,   2, 126,   2,  90, 121, 118, 135, 112, 123, 107,  11,
          1,   1,   1,  93, 112, 123, 112, 109, 121, 111,   1,   1,  93, 121,
        137,   2, 124, 112, 116,   2, 108, 121, 116,   9,   2, 109,   2, 117,
        121, 125, 121, 123, 121, 119,   2,  96, 121, 118, 115,   2, 121, 111,
        121, 118, 112, 118,   9,   1,  81, 111, 112,   2, 122, 121, 110, 115,
        108,   2, 120, 112,   2

In [16]:
n = int(0.9 * len(data))
train_data = data[:n]
val_data = data[n:]

In [19]:
context_window = 8
train_data[:context_window + 1]

tensor([ 93, 112, 123, 112, 109, 121, 111,   1,   1])

In [21]:
x = train_data[:context_window]
y = train_data[1:context_window + 1]

for t in range(context_window):
    context = x[:t+1]
    target = y[t]
    print(f"When input is {context}, prediction is {target}")

When input is tensor([93]), prediction is 112
When input is tensor([ 93, 112]), prediction is 123
When input is tensor([ 93, 112, 123]), prediction is 112
When input is tensor([ 93, 112, 123, 112]), prediction is 109
When input is tensor([ 93, 112, 123, 112, 109]), prediction is 121
When input is tensor([ 93, 112, 123, 112, 109, 121]), prediction is 111
When input is tensor([ 93, 112, 123, 112, 109, 121, 111]), prediction is 1
When input is tensor([ 93, 112, 123, 112, 109, 121, 111,   1]), prediction is 1


In [25]:
torch.manual_seed(1337)
batch_size = 4
conext_window = 8

def get_batch(split):
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - context_window, (batch_size,))
    x = torch.stack([data[i:i+context_window] for i in ix])
    y = torch.stack([data[i+1:i+context_window+1] for i in ix])
    return x, y

xb, yb = get_batch('train')
print('inputs')
print(xb.shape)
print(xb)
print('targets')
print(yb.shape)
print(yb)

print('==========')

for b in range(batch_size):
    for t in range(context_window):
        context = xb[b, :t+1]
        target = yb[b,t]
        print(f"input {context.tolist()} -> target {target}")

inputs
torch.Size([4, 8])
tensor([[  9,   1,  81, 111, 112,   2, 109,   2],
        [114, 115, 125, 135,   9,   1, 101, 125],
        [112, 114, 120, 134, 116,   2, 113, 112],
        [  9,   2, 107,   2, 125, 107, 119,   2]])
targets
torch.Size([4, 8])
tensor([[  1,  81, 111, 112,   2, 109,   2, 125],
        [115, 125, 135,   9,   1, 101, 125, 121],
        [114, 120, 134, 116,   2, 113, 112, 120],
        [  2, 107,   2, 125, 107, 119,   2, 126]])
input [9] -> target 1
input [9, 1] -> target 81
input [9, 1, 81] -> target 111
input [9, 1, 81, 111] -> target 112
input [9, 1, 81, 111, 112] -> target 2
input [9, 1, 81, 111, 112, 2] -> target 109
input [9, 1, 81, 111, 112, 2, 109] -> target 2
input [9, 1, 81, 111, 112, 2, 109, 2] -> target 125
input [114] -> target 115
input [114, 115] -> target 125
input [114, 115, 125] -> target 135
input [114, 115, 125, 135] -> target 9
input [114, 115, 125, 135, 9] -> target 1
input [114, 115, 125, 135, 9, 1] -> target 101
input [114, 115, 125, 135, 